In [2]:
!pip install transformers datasets torch --quiet

In [3]:
import torch
import numpy as np
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
from torch.optim import AdamW
from transformers import get_scheduler

In [4]:
# Small domain-specific dataset: Movie Review Sentiment
texts = [
    "this movie was absolutely brilliant and inspiring",
    "the film had outstanding performances by all actors",
    "a masterpiece of modern cinema truly breathtaking",
    "loved every moment of this wonderful movie",
    "the story was beautifully crafted and deeply moving",
    "incredible visuals and a powerful emotional storyline",
    "one of the best films i have ever watched",
    "the direction and screenplay were simply outstanding",
    "a heartwarming tale that touched my soul deeply",
    "brilliant acting and a gripping plot throughout",
    "this movie was terrible and extremely boring",
    "worst film i have ever seen in my life",
    "the plot made no sense and acting was poor",
    "a complete waste of time and money seriously",
    "dull storyline with no interesting characters at all",
    "the movie dragged on forever with no purpose",
    "disappointing film that failed to deliver anything good",
    "horrible screenplay and very weak character development",
    "the worst cinema experience i have ever had",
    "boring predictable and completely unoriginal storyline",
]

# 1 = Positive, 0 = Negative
labels = [1,1,1,1,1,1,1,1,1,1,
          0,0,0,0,0,0,0,0,0,0]

print(f"Total samples : {len(texts)}")
print(f"Positive      : {labels.count(1)}")
print(f"Negative      : {labels.count(0)}")

Total samples : 20
Positive      : 10
Negative      : 10


In [5]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

encodings = tokenizer(
    texts,
    truncation    = True,
    padding       = True,
    max_length    = 32,
    return_tensors= 'pt'
)

print("Input IDs shape     :", encodings['input_ids'].shape)
print("Attention mask shape:", encodings['attention_mask'].shape)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Input IDs shape     : torch.Size([20, 12])
Attention mask shape: torch.Size([20, 12])


In [6]:
class MovieDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

dataset    = MovieDataset(encodings, labels)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

print(f"Dataset size  : {len(dataset)}")
print(f"Batches       : {len(dataloader)}")

Dataset size  : 20
Batches       : 5


In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels = 2        # Positive / Negative
)
model.to(device)
print("BERT model loaded successfully")

Using device: cpu


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BERT model loaded successfully


In [8]:
optimizer  = AdamW(model.parameters(), lr=2e-5)
num_epochs = 5
num_steps  = num_epochs * len(dataloader)

scheduler  = get_scheduler(
    "linear",
    optimizer          = optimizer,
    num_warmup_steps   = 0,
    num_training_steps = num_steps
)

print(f"Total training steps: {num_steps}")

Total training steps: 25


In [9]:
model.train()

for epoch in range(num_epochs):
    total_loss = 0
    correct    = 0
    total      = 0

    for batch in dataloader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels_batch   = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(
            input_ids      = input_ids,
            attention_mask = attention_mask,
            labels         = labels_batch
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds       = torch.argmax(outputs.logits, dim=1)
        correct    += (preds == labels_batch).sum().item()
        total      += labels_batch.size(0)

    avg_loss = total_loss / len(dataloader)
    accuracy = correct / total
    print(f"Epoch {epoch+1}/{num_epochs}  |  "
          f"Loss: {avg_loss:.4f}  |  Accuracy: {accuracy:.4f}")

Epoch 1/5  |  Loss: 0.7990  |  Accuracy: 0.4500
Epoch 2/5  |  Loss: 0.5779  |  Accuracy: 0.8000
Epoch 3/5  |  Loss: 0.5717  |  Accuracy: 0.8500
Epoch 4/5  |  Loss: 0.4804  |  Accuracy: 0.8500
Epoch 5/5  |  Loss: 0.4814  |  Accuracy: 0.8000


In [10]:
def predict_sentiment(text):
    model.eval()
    inputs = tokenizer(
        text,
        return_tensors = 'pt',
        truncation     = True,
        padding        = True,
        max_length     = 32
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    pred  = torch.argmax(outputs.logits, dim=1).item()
    label = "POSITIVE 😊" if pred == 1 else "NEGATIVE 😞"
    print(f"\nReview  : {text}")
    print(f"Sentiment: {label}")

predict_sentiment("this film was absolutely amazing and wonderful")
predict_sentiment("the movie was dull boring and a waste of time")
predict_sentiment("brilliant performances and a gripping storyline")
predict_sentiment("terrible acting and a very disappointing ending")



Review  : this film was absolutely amazing and wonderful
Sentiment: POSITIVE 😊

Review  : the movie was dull boring and a waste of time
Sentiment: NEGATIVE 😞

Review  : brilliant performances and a gripping storyline
Sentiment: POSITIVE 😊

Review  : terrible acting and a very disappointing ending
Sentiment: NEGATIVE 😞
